---
title: "Cost-per-click surge attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. We use mix-rate decomposition to asses the contribution of each individual product to a global surge on cost per click. In order to get bootstrap confidence intervals on each product contribution we rely on Monte Carlo methods.
format:
  html:
    code-fold: true
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
---

**Revised narrative arc**
Given the 88% mix / 11% rate split, I'd restructure the notebook slightly:

1. **Business problem:** 18% aggregate CPC increase, stakeholders are alarmed
2. **Naive approach:** look at product-level CPC changes — most products look fine, some even improved. Paradox: how can the aggregate be +18%?
3. **Simpson's Paradox:** explain with the N=2 toy example in CPC terms
4. **Shapley attempt:** natural instinct to attribute fairly — show it gives counterintuitive results and explain why (cross-sectional, not causal)
5. **Rate-mix decomposition:** the right framing. Show the 88/11 split — the problem is almost entirely compositional, not a pricing problem
6. **Attribution of the mix effect:** which products contributed most to the click share shift toward expensive inventory? This is your ranked list for stakeholders
7. **Uncertainty quantification:** bootstrap CIs on mix contributions — which attributions are robust at N=1900?

**The key message for stakeholders**
The actionable conclusion changes completely depending on which decomposition you use:

 - Naive / Shapley → "these products got more expensive, fix their CPCs"
 - Rate-mix → "click volume shifted toward expensive products, investigate why the allocation changed"

# Initialization

## Imports and settings

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [ ]:
# Configuration
pd.set_option("max_colwidth", None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
accent_colors = ["#009AD7", "#06C4B0", "#FF540A", "#FEC10D", "#E81159", "#002856"]

## Auxiliary functions

In [ ]:
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df["cpc_before"] = df["cost_before"] / df["clicks_before"]
    df["cpc_after"] = df["cost_after"] / df["clicks_after"]
    df["cpc_diff"] = df["cpc_after"] - df["cpc_before"]

    return df

def aggregate_data(df: pd.DataFrame, numeric_only: bool=False) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum(numeric_only=numeric_only)).T
    df_agg["clicks_before"] = df_agg["clicks_before"].astype(int)
    df_agg["clicks_after"] = df_agg["clicks_after"].astype(int)

    return df_agg

def compute_rate_mix_effects(df: pd.DataFrame) -> pd.DataFrame:
    clicks_total_after = df["clicks_after"].sum()
    clicks_total_before = df["clicks_before"].sum()
    df["rate_effect"] = df["clicks_after"] / clicks_total_after * df["cpc_diff"]
    df["mix_effect"] = (df["clicks_after"] / clicks_total_after - df["clicks_before"] / clicks_total_before) * df["cpc_before"]
    df["total_effect"] = df["rate_effect"] + df["mix_effect"]

    return df

# Understanding the business problem

## The problem

In [ ]:
df_toy = pd.DataFrame(
    columns=["id", "cost_before", "clicks_before", "cost_after", "clicks_after"],
    data=[
        ["A", 10000.00, 200, 1500.00, 25],
        ["B",   800.00,  40,  900.00, 40]
    ]
)

df_toy = add_derived_columns(df_toy)
df_toy = compute_rate_mix_effects(df_toy)

df_toy_agg = aggregate_data(df_toy)
df_toy_agg = add_derived_columns(df_toy_agg)

print("Toy example:")
display(df_toy)
print("Aggregated data:")
display(df_toy_agg)

Even in a simple example with only two products we can see how the phenomenon of Simpson's paradox arises:

- Both products individual CPC goes up: +$10 for product A and +$2.50 for product B

- However the aggregated CPC goes down: -$8.08

This example actually illustrates a very common business situation:

- Product A runs under some algorithm that optimizes RPS

- The algorithm discards sales with low return and the final effect is that both revenue and sales decrease, albeit in a way that RPS increases (in other words: the percentual decrease in revenue is less than the percentual decrease in sales)

- When aggregating the data with other products this has a harming impact on the global RPS

In a real case scenario where we have ~1,900 different products instead of just a couple, these interactions become much more complex.

## The initial solution and why it doesn't work

In order to navigate the paradox we borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us each product is a player and the common goal is the aggregated RPS. We want to measure how much each individual product contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result. In our example above, the sum of the Shapley value of A and the Shapley value of B must be -17.95.

*Note:* It is not our intend to provide a detailed account on how Shapley values are computed. In the present section we ask the reader to trust us, while in future more technical sections we assume the reader has enough familiarity with the concept.

In our example we have v(A) = $10.00, v(B) = $2.50, v(A,B) = -$8.08. Then:

- s(A) = 1/2 [v(A,B) + v(A) - v(B)] = -0.29

- s(B) = 1/2 [v(A,B) + v(B) - v(A)] = -7.79

## Real data

In [ ]:
df = pd.read_csv("../assets/cpc_data.csv")

In [ ]:
df = compute_rate_mix_effects(add_derived_columns(df))

In [ ]:
df_agg = add_derived_columns(aggregate_data(df, numeric_only=True))

In [ ]:
prop_rate = (df_agg["rate_effect"].iloc[0] / df_agg["total_effect"].iloc[0])
prop_mix = (df_agg["mix_effect"].iloc[0] / df_agg["total_effect"].iloc[0])
print(prop_rate, prop_mix)

In [ ]:
def plot_cpc_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_before"],
            y=df["cpc_after"],
            mode="markers",
            marker_size=4,
            marker_color=accent_colors[0],
            text=df["product_id"],
            name="CPC before/after"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0,80],
            y=[0,80],
            mode="lines",
            line=dict(color="red", dash="dash", width=1),
            name="Same CPC before/after"
        )
    )

    fig.update_layout(
        title="CPC comparison (before vs after)",
        width=600,
        height=600,
        xaxis_title="CPC before",
        yaxis_title="CPC after",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
plot_cpc_comparison(df)

In [ ]:
def plot_mix_rate_effect_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["rate_effect"],
            y=df["mix_effect"],
            mode="markers",
            marker_size=4,
            marker_color=accent_colors[0],
            text=df["product_id"],
            name="Rate-mix effect"
        )
    )

    for i in range(-1,6):
        fig.add_trace(
            go.Scatter(
                x=[-0.02,0.02],
                y=[0.01 * i + 0.02, 0.01 * i - 0.02],
                mode="lines",
                line=dict(color="red", dash="dash", width=1),
                name=f"Total effect = {0.01 * i}"
            )
        )

    fig.update_layout(
        title="Rate vs Mix effect on CPC change",
        width=600,
        height=600,
        xaxis_title="Rate effect",
        yaxis_title="Mix effect",
        #yaxis_scaleanchor="x",
        #yaxis_scaleratio=1,
        xaxis_range=[-0.021, 0.021],
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
# plot_cpc_comparison(df)
plot_mix_rate_effect_comparison(df)

**What to look for now**
The rate effect for product $i$ is $w_i^{\text{after}} \cdot \Delta\text{CPC}_i$. A product contributes strongly to the rate effect if:
- Its click weight is large and it gained a lot of CPC

The mix effect for product $i$ is $\Delta w_i \cdot \text{CPC}_i^{\text{before}}$​. A product contributes strongly to the mix effect if:
- It gained a lot of click share ($\Delta w_i$ is large and positive) and had a high baseline CPC



In [ ]:
def plot_mix_effect_factors(df: pd.DataFrame) -> None:
    ps_weight_delta = df["clicks_after"] / df["clicks_after"].sum() - df["clicks_before"] / df["clicks_before"].sum()
    
    fig = go.Figure()

    max_abs = np.abs(df["mix_effect"]).max()

    fig.add_trace(
        go.Scatter(
            x=ps_weight_delta,
            y=df["cpc_before"],
            mode="markers",
            marker=dict(
                size=5,
                color=df["mix_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Mix effect")
            ),
            text=df.apply(lambda row: f"{row['product_id']} mix effect = {round(row['mix_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Mix effect factors",
        width=600,
        height=600,
        xaxis_title="Weight delta",
        yaxis_title="CPC before",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
def plot_rate_effect_factors(df: pd.DataFrame) -> None:
    ps_weight= df["clicks_after"] / df["clicks_after"].sum()
    
    fig = go.Figure()

    max_abs = np.abs(df["rate_effect"]).max()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_diff"],
            y=ps_weight,
            mode="markers",
            marker=dict(
                size=5,
                color=df["rate_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Rate effect")
            ),
            text=df.apply(lambda row: f"{row['product_id']} rate effect = {round(row['rate_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Rate effect factors",
        width=600,
        height=600,
        xaxis_title="CPC delta",
        yaxis_title="Clicks weight after",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
plot_mix_effect_factors(df)
plot_rate_effect_factors(df)